In [1]:
#apply StandardScaler
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
from sklearn.cluster import OPTICS
from sklearn.cluster import Birch
from sklearn.decomposition import PCA

#load dataset
df= pd.read_csv("data/pain_dataset_200P_4hz.csv")
df

# Drop target and ID column & target column
X = df.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape:", X.shape)

Features shape: (96000, 7)


In [2]:
#Apply StandardScaler
scaler = StandardScaler()
X_sp = scaler.fit_transform(X)

print("Features shape (scaled version):", X_sp.shape)

Features shape (scaled version): (96000, 7)


In [3]:
#apply PCA
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_sp)#Apply PCA
#print("PCA features shape:", X_pca.shape)
print("PCA-reduced features shape:", X_pca.shape)
print("Explained variance ratio sum:", sum(pca.explained_variance_ratio_))

PCA-reduced features shape: (96000, 6)
Explained variance ratio sum: 0.9563496543214839


In [4]:
#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [5]:
#K-Means on Scaled + PCA Data
start_time = time.time()
kmean_pca = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    kmean_pca.append({"algorithm": "KMeans", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")   


Runtime: 1397.439201593399 seconds
K-Means runtime: 1397.4392 seconds


In [6]:
#Gaussian Mixture (GMM)on Scaled + PCA Data
start_time = time.time()
gmm_pca = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    gmm_pca.append({"algorithm": "GMM", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")   

Runtime: 1632.5912644863129 seconds
GMM runtime: 1632.5913 seconds


In [ ]:
# from sklearn.preprocessing import MaxAbsScaler
# scaler = MaxAbsScaler()
# X_max_sp = scaler.fit_transform(X)

# print("Features shape (scaled version):", X_max_sp.shape)

# #apply PCA
# pca = PCA(n_components=0.95, random_state=42)
# X_pca = pca.fit_transform(X_max_sp)#Apply PCA
# #print("PCA features shape:", X_pca.shape)
# print("PCA-reduced features shape:", X_pca.shape)
# X_new = X_pca.astype("float32")
# start_time = time.time()

# agg_pca = []
# for k in k_values:
#     agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
#     labels = agg.fit_predict(X_new)
#     sil, db, ch = compute_metrics(X_new, labels)
#     agg_pca.append({"algorithm": "Agglomerative", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

# end_time = time.time()
# runtime = end_time - start_time
# print("Runtime:", runtime, "seconds")
# print(f"Agglomerative runtime: {runtime:.4f} seconds")   

In [7]:
#Agglomerative Clustering on Scaled + PCA Data
df_small = df.sample(n=5000, random_state=42)
X_small = df_small.drop(columns=["person_ID", "pain_scale"], errors="ignore")
X_small_scaled = scaler.fit_transform(X_small)
X_small_pca = pca.fit_transform(X_small_scaled)

start_time = time.time()
agg_pca = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_small_pca)
    sil, db, ch = compute_metrics(X_small_pca, labels)
    agg_pca.append({"algorithm": "Agglomerative", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")   

Runtime: 12.884352207183838 seconds
Agglomerative runtime: 12.8844 seconds


In [ ]:
print("PCA-reduced for small data:", X_small_pca.shape)

PCA-reduced for small data: (5000, 6)


In [8]:
#Spectral Clustering on Scaled + PCA Data
start_time = time.time()
spec_pca = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_small_pca)
    sil, db, ch = compute_metrics(X_small_pca, labels)
    spec_pca.append({"algorithm": "Spectral", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 38.04187345504761 seconds
Spectral runtime: 38.0419 seconds


In [9]:
#DBSCAN on Scaled + PCA Data
start_time = time.time()
dbscan_pca = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_pca)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 and len(set(labels[mask])) > 1:
        sil, db, ch = compute_metrics(X_pca[mask], labels[mask])
        dbscan_pca.append({"algorithm": "DBSCAN", "preprocessing": "PCA", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 407.85342502593994 seconds
DBSCAN runtime: 407.8534 seconds


In [ ]:
# #OPTICS on Scaled + PCA Data
# start_time = time.time()
# optics_pca = []
# min_samples_values = [3, 5, 10, 20]

# for m in min_samples_values:
#     optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
#     labels = optics.fit_predict(X_pca)

#     # Remove noise points (-1) if needed
#     unique_labels = set(labels) - {-1}

#     if len(unique_labels) > 1:
#         sil, db, ch = compute_metrics(X_pca, labels)
#         optics_pca.append({
#             "algorithm": "OPTICS",
#             "preprocessing": "PCA",
#             "min_samples": m,
#             "xi": 0.05,
#             "n_clusters": len(unique_labels),
#             "silhouette": sil,
#             "davies_bouldin": db,
#             "calinski_harabasz": ch
#         })

# end_time = time.time()
# runtime = end_time - start_time
# print("Runtime:", runtime, "seconds")
# print(f"OPTICS runtime: {runtime:.4f} seconds")

Runtime: 14747.92797780037 seconds
OPTICS runtime: 14747.9280 seconds


In [10]:
#BIRCH on Scaled + PCA Data
start_time = time.time()
birch_pca = []
#threshold_values = [0.2, 0.5, 1.0, 1.5]
threshold_values = [1.5, 3.0, 5.0, 10.0]

for t in threshold_values:
    birch = Birch(n_clusters=None, branching_factor=200,threshold=t)
    labels = birch.fit_predict(X_pca)

    n_clusters = len(set(labels))
    if 1 < n_clusters < len(X_pca):
        sil, db, ch = compute_metrics(X_pca, labels)
        birch_pca.append({
            "algorithm": "BIRCH",
            "preprocessing": "PCA",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds")

Runtime: 182.4823203086853 seconds
BIRCH runtime: 182.4823 seconds


In [11]:
#OPTICS on Scaled + PCA Data with 5k data
start_time = time.time()
optics_pca = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_small_pca)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_small_pca, labels)
        optics_pca.append({
            "algorithm": "OPTICS",
            "preprocessing": "PCA",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"OPTICS runtime: {runtime:.4f} seconds")

Runtime: 491.533034324646 seconds
OPTICS runtime: 491.5330 seconds


In [12]:
print(optics_pca)

[{'algorithm': 'OPTICS', 'preprocessing': 'PCA', 'min_samples': 3, 'xi': 0.05, 'n_clusters': 336, 'silhouette': -0.3763954924260118, 'davies_bouldin': 1.393814778327462, 'calinski_harabasz': 4.598911505816719}, {'algorithm': 'OPTICS', 'preprocessing': 'PCA', 'min_samples': 5, 'xi': 0.05, 'n_clusters': 56, 'silhouette': -0.49085746717797135, 'davies_bouldin': 1.5535654626132234, 'calinski_harabasz': 4.948092727680692}, {'algorithm': 'OPTICS', 'preprocessing': 'PCA', 'min_samples': 10, 'xi': 0.05, 'n_clusters': 2, 'silhouette': -0.30600795458178454, 'davies_bouldin': 1.8306391670441764, 'calinski_harabasz': 10.373767983232016}]


In [ ]:
import csv

pain_results_pca = (kmean_pca + gmm_pca + agg_pca + spec_pca + dbscan_pca + birch_pca + optics_pca)

keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

#with open('updated_data/pain_data/pain_pca.csv', 'w', newline='') as file:
with open('updated_data/pain_new_data/pain_pca.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(pain_results_pca)


In [13]:
from sklearn.metrics import adjusted_rand_score
import numpy as np
import pandas as pd


# ARI settings
n_bootstrap = 100
ari_results = []


# Helper function

def fit_and_predict(name, params, X_data):

    if name == "K-Means":
        model = KMeans(n_clusters=params["k"],n_init=n_init,random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "GMM":
        model = GaussianMixture(n_components=params["k"],n_init=n_init,random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "Agglomerative":
        model = AgglomerativeClustering(n_clusters=params["k"],linkage='ward')
        labels = model.fit_predict(X_data)

    elif name == "Spectral":
        model = SpectralClustering(n_clusters=params["k"],affinity='nearest_neighbors',n_init=n_init,random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "DBSCAN":
        model = DBSCAN(eps=params["eps"],min_samples=min_samples)
        labels = model.fit_predict(X_data)

    elif name == "BIRCH":
        model = Birch(n_clusters=None,threshold=params["threshold"])
        labels = model.fit_predict(X_data)

    elif name == "OPTICS":
        model = OPTICS(min_samples=params["min_samples"],xi=0.05,n_jobs=-1)
        labels = model.fit_predict(X_data)

    else:
        return None

    return labels



# Function for bootstrap ARI

def bootstrap_ari(algo_name, params, X_data):

    print(f"Processing: {algo_name} {params}")
    # Reproducible bootstrap sampling
    rng = np.random.RandomState(42)

    # Reference clustering on full dataset
    ref_labels = fit_and_predict(algo_name,params,X_data)

    if ref_labels is None:
        return None

    ari_scores = []

    for b in range(n_bootstrap):

        # Bootstrap sample
        indices = rng.choice(len(X_data),size=len(X_data),replace=True)

        X_boot = X_data[indices]

        # Cluster bootstrap sample
        boot_labels = fit_and_predict(algo_name,params,X_boot)

        if boot_labels is None:
            continue

        # Reference labels for sampled observations
        ref_subset = np.asarray(ref_labels)[indices]

        # ARI
        ari = adjusted_rand_score(ref_subset,np.asarray(boot_labels))

        ari_scores.append(ari)

    if len(ari_scores) == 0:
        return None

    return {
        "algorithm": algo_name,
        "preprocessing": "PCA",
        **params,
        "ARI_mean": np.mean(ari_scores),
        "ARI_std": np.std(ari_scores),
        "stability_score": np.mean(ari_scores) - np.std(ari_scores),
        "n_bootstrap": len(ari_scores)
    }

In [14]:
for r in kmean_pca:

    result = bootstrap_ari(
        "K-Means",
        {"k": r["k"]},
        X_pca
    )

    if result is not None:
        ari_results.append(result)

print("K-Means completed.")

Processing: K-Means {'k': 2}
Processing: K-Means {'k': 3}
Processing: K-Means {'k': 4}
Processing: K-Means {'k': 5}
Processing: K-Means {'k': 6}
Processing: K-Means {'k': 7}
Processing: K-Means {'k': 8}
K-Means completed.


In [15]:
for r in gmm_pca:

    result = bootstrap_ari(
        "GMM",
        {"k": r["k"]},
        X_pca
    )

    if result is not None:
        ari_results.append(result)


print("GMM completed.")

Processing: GMM {'k': 2}
Processing: GMM {'k': 3}
Processing: GMM {'k': 4}
Processing: GMM {'k': 5}
Processing: GMM {'k': 6}
Processing: GMM {'k': 7}
Processing: GMM {'k': 8}
GMM completed.


In [16]:
for r in agg_pca:

    result = bootstrap_ari(
        "Agglomerative",
        {"k": r["k"]},
        X_small_pca
    )

    if result is not None:
        ari_results.append(result)

print("Agglomerative completed.")

Processing: Agglomerative {'k': 2}
Processing: Agglomerative {'k': 3}
Processing: Agglomerative {'k': 4}
Processing: Agglomerative {'k': 5}
Processing: Agglomerative {'k': 6}
Processing: Agglomerative {'k': 7}
Processing: Agglomerative {'k': 8}
Agglomerative completed.


In [17]:
for r in spec_pca:

    result = bootstrap_ari(
        "Spectral",
        {"k": r["k"]},
        X_small_pca
    )

    if result is not None:
        ari_results.append(result)

print("Spectral completed.")

Processing: Spectral {'k': 2}


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Processing: Spectral {'k': 3}


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Processing: Spectral {'k': 4}


...

Processing: Spectral {'k': 7}


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Processing: Spectral {'k': 8}


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Spectral completed.


In [18]:
for r in dbscan_pca:

    result = bootstrap_ari(
        "DBSCAN",
        {"eps": r["eps"]},
        X_pca
    )

    if result is not None:
        ari_results.append(result)

print("DBSCAN completed.")

Processing: DBSCAN {'eps': 0.5}
Processing: DBSCAN {'eps': 1.0}
DBSCAN completed.


In [19]:
for r in birch_pca:

    result = bootstrap_ari(
        "BIRCH",
        {"threshold": r["threshold"]},
        X_pca
    )

    if result is not None:
        ari_results.append(result)

print("BIRCH completed.")

Processing: BIRCH {'threshold': 1.5}
BIRCH completed.


In [20]:
for r in optics_pca:

    result = bootstrap_ari(
        "OPTICS",
        {"min_samples": r["min_samples"]},
        X_small_pca
    )

    if result is not None:
        ari_results.append(result)

print("OPTICS completed.")

Processing: OPTICS {'min_samples': 3}


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden

Processing: OPTICS {'min_samples': 5}


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden

Processing: OPTICS {'min_samples': 10}
OPTICS completed.


In [21]:

# Final ARI summary


ari_df = pd.DataFrame(ari_results).round(4)

print("\n BOOTSTRAP ARI STABILITY ")
print(ari_df.to_string(index=False))


# Top 3 by ARI mean
top3_ari = ari_df.nlargest(
    3,
    "ARI_mean"
)

print("\n TOP 3 MOST STABLE BY ARI ")
print(top3_ari.to_string(index=False))


# Save results
ari_df.to_csv("updated_data/ARI_Score/pain_pca_ari.csv", index=False)



 BOOTSTRAP ARI STABILITY 
    algorithm preprocessing   k  ARI_mean  ARI_std  stability_score  n_bootstrap  eps  threshold  min_samples
      K-Means           PCA 2.0    0.9951   0.0022           0.9929          100  NaN        NaN          NaN
      K-Means           PCA 3.0    0.9707   0.0171           0.9536          100  NaN        NaN          NaN
      K-Means           PCA 4.0    0.8783   0.0611           0.8172          100  NaN        NaN          NaN
      K-Means           PCA 5.0    0.7749   0.1005           0.6744          100  NaN        NaN          NaN
      K-Means           PCA 6.0    0.6681   0.1312           0.5369          100  NaN        NaN          NaN
      K-Means           PCA 7.0    0.5774   0.1514           0.4260          100  NaN        NaN          NaN
      K-Means           PCA 8.0    0.5356   0.1372           0.3984          100  NaN        NaN          NaN
          GMM           PCA 2.0    0.9903   0.0030           0.9872          100  NaN        

In [22]:
#the top three by stability score
ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

top_3 = (
    ari_df
    .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
    .head(3)
    .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
)

print(top_3.round(3).to_string(index=False))

algorithm   k  ARI_mean  ARI_std  Stability Score
  K-Means 2.0     0.995    0.002            0.996
      GMM 2.0     0.990    0.003            0.994
   DBSCAN NaN     0.623    0.005            0.990


In [23]:
#the best three clustering results overall, sort primarily by ARI_mean
top_3 = ari_df.sort_values(
    ["ARI_mean", "Stability Score"],
    ascending=[False, False]
).head(3)

print(top_3.round(3).to_string(index=False))

algorithm preprocessing   k  ARI_mean  ARI_std  stability_score  n_bootstrap  eps  threshold  min_samples  Stability Score
  K-Means           PCA 2.0     0.995    0.002            0.993          100  NaN        NaN          NaN            0.996
      GMM           PCA 2.0     0.990    0.003            0.987          100  NaN        NaN          NaN            0.994
      GMM           PCA 3.0     0.982    0.005            0.976          100  NaN        NaN          NaN            0.989


In [ ]:
# ari_df.to_csv("updated_data/ARI_Score/pain_pca_ari.csv", index=False)

In [25]:

# TOP 3 RESULTS FOR EACH ALGORITHM INDIVIDUALLY


all_algorithms = {
    "K-Means": kmean_pca,
    "GMM": gmm_pca,
    "Agglomerative": agg_pca,
    "Spectral": spec_pca,
    "DBSCAN": dbscan_pca,
    "BIRCH": birch_pca,
    "OPTICS": optics_pca
}



for algorithm, results in all_algorithms.items():

    if len(results) == 0:
        continue

    result_df = pd.DataFrame(results)


    # Round all validation values to 4 decimal places

    result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ] = result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ].round(4)



    print("\n")
   
    print(algorithm)
   



    # Select parameter column
  
    parameter_columns = [
        "k",
        "eps",
        "threshold",
        "min_samples"
    ]


    parameter = None

    for col in parameter_columns:
        if col in result_df.columns:
            parameter = col
            break



    # TOP 3 SILHOUETTE


    print("\nTop 3 Silhouette Score (Higher is better)")

    top_sil = result_df.nlargest(
        3,
        "silhouette"
    )

    print(
        top_sil[
            [
                parameter,
                "silhouette"
            ]
        ].to_string(index=False)
    )




    # TOP 3 DAVIES-BOULDIN
  

    print("\nTop 3 Davies-Bouldin Index (Lower is better)")

    top_db = result_df.nsmallest(
        3,
        "davies_bouldin"
    )

    print(
        top_db[
            [
                parameter,
                "davies_bouldin"
            ]
        ].to_string(index=False)
    )



    # TOP 3 CALINSKI-HARABASZ
   

    print("\nTop 3 Calinski-Harabasz Index (Higher is better)")

    top_ch = result_df.nlargest(
        3,
        "calinski_harabasz"
    )

    print(
        top_ch[
            [
                parameter,
                "calinski_harabasz"
            ]
        ].to_string(index=False)
    )



K-Means

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.2349
 3      0.1593
 4      0.1483

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.5971
 8          1.7949
 7          1.8602

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2         34438.4601
 3         23529.1822
 4         19122.8834


GMM

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.2315
 3      0.1595
 4      0.1233

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.6119
 3          2.0385
 5          2.2781

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2         33590.7740
 3         22944.6336
 4         18190.7329


Agglomerative

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1856
 3      0.1002
 4      0.0886

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.7725
 6          2.2305
 5          2.2712

Top 3 Calinski-H

In [26]:
#  Combine all algorithm results 
all_results = (
    kmean_pca +
    gmm_pca +
    agg_pca +
    spec_pca +
    dbscan_pca +
    birch_pca +
    optics_pca
)

results_df = pd.DataFrame(all_results)

# Round metric values to 4 decimal places
metric_cols = ["silhouette", "davies_bouldin", "calinski_harabasz"]
results_df[metric_cols] = results_df[metric_cols].round(4)

# Columns that may exist depending on algorithm
possible_cols = ["algorithm", "k", "eps", "threshold", "min_samples","n_clusters"]

def available_cols(df, metric):
    cols = [c for c in possible_cols if c in df.columns]
    cols.append(metric)
    return cols

#  Top 3 by Silhouette (higher is better) 
top3_sil = results_df.nlargest(3, "silhouette")

print("\nTOP 3 SILHOUETTE")
print(top3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

# Top 3 by Davies-Bouldin (lower is better) 
top3_db = results_df.nsmallest(3, "davies_bouldin")

print("\nTOP 3 DAVIES-BOULDIN ")
print(top3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Top 3 by Calinski-Harabasz (higher is better) 
top3_ch = results_df.nlargest(3, "calinski_harabasz")

print("\n TOP 3 CALINSKI-HARABASZ ")
print(top3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))

#Bottom 3 by Silhouette (lower is worse) 
bottom3_sil = results_df.nsmallest(3, "silhouette")

print("\nBOTTOM 3 SILHOUETTE ")
print(bottom3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

# Bottom 3 by Davies-Bouldin (higher is worse) 
bottom3_db = results_df.nlargest(3, "davies_bouldin")

print("\n BOTTOM 3 DAVIES-BOULDIN ")
print(bottom3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Bottom 3 by Calinski-Harabasz (lower is worse) 
bottom3_ch = results_df.nsmallest(3, "calinski_harabasz")

print("\n BOTTOM 3 CALINSKI-HARABASZ ")
print(bottom3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))


TOP 3 SILHOUETTE
algorithm   k  eps  threshold  min_samples  n_clusters  silhouette
   DBSCAN NaN  1.0        NaN          NaN         NaN      0.3608
   KMeans 2.0  NaN        NaN          NaN         NaN      0.2349
      GMM 2.0  NaN        NaN          NaN         NaN      0.2315

TOP 3 DAVIES-BOULDIN 
algorithm   k  eps  threshold  min_samples  n_clusters  davies_bouldin
   DBSCAN NaN  1.0        NaN          NaN         NaN          0.5965
   DBSCAN NaN  0.5        NaN          NaN         NaN          0.9510
   OPTICS NaN  NaN        NaN          3.0       336.0          1.3938

 TOP 3 CALINSKI-HARABASZ 
algorithm   k  eps  threshold  min_samples  n_clusters  calinski_harabasz
   KMeans 2.0  NaN        NaN          NaN         NaN         34438.4601
      GMM 2.0  NaN        NaN          NaN         NaN         33590.7740
   KMeans 3.0  NaN        NaN          NaN         NaN         23529.1822

BOTTOM 3 SILHOUETTE 
algorithm   k  eps  threshold  min_samples  n_clusters  silhou